In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Przygotowanie danych
transform = transforms.Compose([
    transforms.ToTensor(),  # Przekształcenie obrazów na tensory
    transforms.Normalize((0.5,), (0.5,))  # Normalizacja do zakresu [-1, 1]
])

# Pobranie danych MNIST
train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

# Ładowanie danych
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# 2. Definicja modelu
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()  # Przekształcenie obrazu 28x28 na wektor 784
        self.fc1 = nn.Linear(28 * 28, 512)  # Warstwa ukryta
        self.fc2 = nn.Linear(512, 10)  # Warstwa wyjściowa
        
    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.softmax(self.fc2(x), dim=1)
        return x

model = NeuralNetwork()

# 3. Konfiguracja procesu uczenia
criterion = nn.CrossEntropyLoss()  # Funkcja straty
optimizer = optim.RMSprop(model.parameters(), lr=0.001)  # Optymalizator RMSprop

# 4. Trenowanie modelu
epochs = 5

for epoch in range(epochs):
    model.train()  # Tryb trenowania
    running_loss = 0.0
    
    for images, labels in train_loader:
        optimizer.zero_grad()  # Wyzerowanie gradientów
        outputs = model(images)  # Przepuszczenie danych przez model
        loss = criterion(outputs, labels)  # Obliczenie straty
        loss.backward()  # Backpropagation
        optimizer.step()  # Aktualizacja wag
        
        running_loss += loss.item()
    
    print(f"Epoka {epoch + 1}/{epochs}, Strata: {running_loss / len(train_loader)}")

# 5. Walidacja modelu
model.eval()  # Tryb ewaluacji
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)  # Pobranie klasy z największym prawdopodobieństwem
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Dokładność na zbiorze testowym: {100 * correct / total:.2f}%")
